# 00 - Data Audit Notebook

This notebook provides a comprehensive audit of the Stats360/StatsBomb data before training.

## Contents

1. **Data Loading**: Load events from StatsBomb Open Data
2. **Coverage Analysis**: Competitions, matches, players covered
3. **Event Distribution**: Types, frequencies, quality
4. **Player Activity**: Events per player, minutes played
5. **Freeze Frame Analysis**: 360 data availability
6. **Data Quality Checks**: Missing values, outliers

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import json
import requests
from pathlib import Path

# Project imports
import sys
sys.path.append('..')

from datasets.schema_contracts import EVENT_TYPE_CATEGORIES, POSITION_CATEGORIES

# Configuration
BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

print("Imports loaded successfully!")

## 1. Load Competitions

In [ ]:
def load_competitions():
    """Load available competitions."""
    url = f"{BASE_URL}/competitions.json"
    response = requests.get(url)
    return pd.DataFrame(response.json())

def load_matches(competition_id, season_id):
    """Load matches for a competition/season."""
    url = f"{BASE_URL}/matches/{competition_id}/{season_id}.json"
    response = requests.get(url)
    return pd.DataFrame(response.json())

# Load competitions
print("Loading competitions...")
competitions = load_competitions()
print(f"Found {len(competitions)} competition/season combinations")

# Display available competitions
competitions_summary = competitions.groupby('competition_name').agg({
    'season_name': 'count',
}).rename(columns={'season_name': 'n_seasons'})

print("\nAvailable Competitions:")
competitions_summary.sort_values('n_seasons', ascending=False).head(15)

## 2. Load Sample Matches

In [ ]:
# Select competitions for analysis
TARGET_COMPETITIONS = [
    (11, 27, "La Liga 2015/16"),
    (43, 3, "World Cup 2018"),
]

all_matches = []
for comp_id, season_id, name in TARGET_COMPETITIONS:
    try:
        matches = load_matches(comp_id, season_id)
        matches['competition_name'] = name
        all_matches.append(matches)
        print(f"  {name}: {len(matches)} matches")
    except Exception as e:
        print(f"  {name}: Failed to load - {e}")

if all_matches:
    matches_df = pd.concat(all_matches, ignore_index=True)
    print(f"\nTotal matches: {len(matches_df)}")

## 3. Event Distribution Analysis

In [ ]:
# Load events for sample matches
def load_events(match_id):
    """Load events for a match."""
    url = f"{BASE_URL}/events/{match_id}.json"
    response = requests.get(url)
    return response.json()

# Sample 10 matches for analysis
sample_matches = matches_df['match_id'].iloc[:10].tolist() if 'matches_df' in dir() else []

all_events = []
for mid in sample_matches:
    try:
        events = load_events(mid)
        all_events.extend(events)
        print(f"  Match {mid}: {len(events)} events")
    except:
        pass

print(f"\nLoaded {len(all_events)} events from {len(sample_matches)} matches")

In [ ]:
# Event type distribution
event_types = defaultdict(int)
for e in all_events:
    event_type = e.get('type', {}).get('name', 'Unknown')
    event_types[event_type] += 1

# Plot distribution
event_df = pd.DataFrame([
    {'Event Type': k, 'Count': v} 
    for k, v in sorted(event_types.items(), key=lambda x: -x[1])[:20]
])

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=event_df, x='Count', y='Event Type', ax=ax, palette='viridis')
ax.set_title('Event Type Distribution (Top 20)', fontsize=14)
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 4. Player Activity Analysis

In [ ]:
# Extract player data
player_events = defaultdict(list)
for e in all_events:
    player = e.get('player', {})
    if player:
        pid = player.get('id')
        pname = player.get('name')
        if pid:
            player_events[pid].append({
                'name': pname,
                'type': e.get('type', {}).get('name'),
                'position': e.get('position', {}).get('name'),
            })

# Compute stats per player
player_stats = []
for pid, events in player_events.items():
    player_stats.append({
        'player_id': pid,
        'player_name': events[0]['name'],
        'n_events': len(events),
        'n_unique_types': len(set(e['type'] for e in events)),
        'positions': list(set(e['position'] for e in events if e['position'])),
    })

player_stats_df = pd.DataFrame(player_stats)
print(f"Player Activity Summary:")
print(f"  Total players: {len(player_stats_df)}")
print(f"  Events per player: mean={player_stats_df['n_events'].mean():.1f}, median={player_stats_df['n_events'].median():.1f}")

# Top players by activity
player_stats_df.sort_values('n_events', ascending=False).head(10)

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(player_stats_df['n_events'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Events')
axes[0].set_ylabel('Number of Players')
axes[0].set_title('Events per Player Distribution')
axes[0].axvline(player_stats_df['n_events'].median(), color='r', linestyle='--', label='Median')
axes[0].legend()

axes[1].hist(player_stats_df['n_unique_types'], bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Number of Unique Event Types')
axes[1].set_ylabel('Number of Players')
axes[1].set_title('Event Type Diversity per Player')

plt.tight_layout()
plt.show()

## 5. Location & 360 Data Quality

In [ ]:
# Check location availability
events_with_location = sum(1 for e in all_events if e.get('location'))
print(f"Location Data:")
print(f"  Events with location: {events_with_location:,} ({events_with_location/len(all_events)*100:.1f}%)")

# Check freeze frame (360) data
events_with_360 = sum(1 for e in all_events if e.get('shot', {}).get('freeze_frame') or 
                      e.get('pass', {}).get('freeze_frame') or
                      e.get('carry', {}).get('freeze_frame'))
print(f"  Events with 360 data: {events_with_360:,} ({events_with_360/len(all_events)*100:.1f}%)")

# Location heatmap
locations = [(e['location'][0], e['location'][1]) for e in all_events if e.get('location')]
if locations:
    x_coords, y_coords = zip(*locations)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    hb = ax.hexbin(x_coords, y_coords, gridsize=20, cmap='YlOrRd', mincnt=1)
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)
    ax.set_xlabel('X (Pitch Length)')
    ax.set_ylabel('Y (Pitch Width)')
    ax.set_title('Event Location Heatmap')
    ax.set_aspect('equal')
    plt.colorbar(hb, ax=ax, label='Event Count')
    plt.tight_layout()
    plt.show()

## 6. Data Quality Summary

In [ ]:
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)
print(f"Total events: {len(all_events):,}")
print(f"Unique players: {len(player_events):,}")
print(f"Events per player (mean): {np.mean([len(v) for v in player_events.values()]):.1f}")
print(f"Location coverage: {events_with_location/len(all_events)*100:.1f}%")
print(f"360 data coverage: {events_with_360/len(all_events)*100:.1f}%")
print(f"\nEvent types: {len(event_types)}")
print(f"Top 5 event types: {list(dict(sorted(event_types.items(), key=lambda x: -x[1])[:5]).keys())}")
print("="*60)
print("\n✅ Data audit complete!")